# Welles — Colab training (run top to bottom)

Ignore every old cell from chat. This notebook is the whole recipe.

**Start clean (do this once)**
1. Runtime → Disconnect and delete runtime → Yes.
2. Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save.
3. Run the cells below **in order**. Wait for a green check before the next one.
4. When cell 6 finishes, run cell 7 **immediately** — do not Restart or Disconnect until the Hub upload prints a link.

On your Desktop you need `Writer_ai.zip` and your Hugging Face write token  
(from `Welles-HuggingFace-Token.txt` on your PC — paste it when asked, never type it into a cell).

## 1. Confirm the GPU
You want `cuda: True` and `Tesla T4`.

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU, then Runtime → Disconnect and delete runtime, then come back.")

## 2. Install training packages
This can sit still for a few minutes. Wait for the green check.

In [ ]:
%pip install -q transformers datasets accelerate peft trl huggingface_hub bitsandbytes
import trl
print("trl ok", trl.__version__)

## 3. Upload the project zip
When **Choose Files** appears: Desktop → **Writer_ai.zip** (not the notebook, not the token file).

You want: `project root: /content`

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

print("Click Choose Files → Desktop → Writer_ai.zip")
uploaded = files.upload()

for name in uploaded:
    src = Path("/content") / name
    if src.suffix.lower() != ".zip":
        print("skipped (not a zip):", name)
        continue
    print("unpacking", src)
    with zipfile.ZipFile(src) as zf:
        for info in zf.infolist():
            rel = info.filename.replace("\\", "/")
            if rel.endswith("/") or not rel:
                continue
            dest = Path("/content") / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as fh, open(dest, "wb") as out:
                out.write(fh.read())

hits = [p for p in Path("/content").glob("**/scripts/train.py") if ".venv" not in str(p)]
if not hits:
    raise SystemExit("No scripts/train.py after unpack. You must pick Writer_ai.zip from the Desktop.")
ROOT = hits[0].parent.parent
print("project root:", ROOT)
print("train.py:", hits[0])

## 4. Hugging Face login
Paste your **write** token in the hidden box, then Enter. You should see `Login successful`.

In [ ]:
from huggingface_hub import login
from getpass import getpass

login(token=getpass("Hugging Face write token (hidden): "))

## 5. Build the training file
Downloads LongWriter and writes `data/train.jsonl`. Wait for it to finish.
You want a line like `wrote .../data/train.jsonl` and a number of examples.

In [ ]:
from pathlib import Path
import os

hits = [p for p in Path("/content").glob("**/scripts/train.py") if ".venv" not in str(p)]
ROOT = hits[0].parent.parent
os.chdir(ROOT)
print("project root:", ROOT)

!python scripts/prepare_data.py --max-examples 400 --max-words 2500

data = ROOT / "data" / "train.jsonl"
print("train.jsonl exists:", data.exists(), data)

## 6. Train + upload (one cell)
Paste your HF token when asked **at the start**, then leave the tab open.
This cell trains, checks the files exist, then uploads to `n0social/welles` by itself.
Done only when you see `https://huggingface.co/n0social/welles`.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
from getpass import getpass
from datasets import load_dataset
from huggingface_hub import HfApi, login
from peft import LoraConfig
from transformers import BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
import torch

# Login FIRST so upload can run unattended after training.
login(token=getpass("Hugging Face write token (hidden): "))

hits = [p for p in Path("/content").glob("**/scripts/train.py") if ".venv" not in str(p)]
if not hits:
    raise SystemExit("Project missing. Run the zip upload cell first.")
ROOT = hits[0].parent.parent
os.chdir(ROOT)
data = ROOT / "data" / "train.jsonl"
out = ROOT / "outputs" / "welles"
adapter_dir = out / "adapter"
if not data.exists():
    raise SystemExit("Missing data/train.jsonl — run the prepare_data cell first.")

free_gb = round(torch.cuda.mem_get_info()[0] / 1e9, 2)
print("VRAM free GB:", free_gb)
if free_gb < 10:
    raise SystemExit("GPU still busy. Runtime → Restart session, then run cells 1–5, then this cell.")

dataset = load_dataset("json", data_files=str(data), split="train")
print(len(dataset), "examples")

trainer = SFTTrainer(
    model="Qwen/Qwen3-8B",
    args=SFTConfig(
        output_dir=str(out),
        learning_rate=2e-4,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=1024,
        logging_steps=5,
        save_steps=50,
        save_total_limit=2,
        fp16=False,
        bf16=False,
        optim="paged_adamw_8bit",
        assistant_only_loss=True,
        report_to="none",
        model_init_kwargs={
            "dtype": torch.float16,
            "attn_implementation": "sdpa",
            "device_map": {"": 0},
        },
    ),
    train_dataset=dataset,
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
    ),
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    ),
)
trainer.train()
adapter_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(adapter_dir))
files = sorted(p.name for p in adapter_dir.glob("*"))
print("saved", adapter_dir)
print("files:", files)
if not files:
    raise SystemExit("save_model wrote nothing. Stop and paste this output.")

api = HfApi()
api.create_repo("n0social/welles", repo_type="model", exist_ok=True, private=False)
api.upload_folder(folder_path=str(adapter_dir), repo_id="n0social/welles", repo_type="model")
print("DONE https://huggingface.co/n0social/welles")

## 7. (Only if cell 6 saved but upload failed)
Skip this if cell 6 already printed the DONE link.

In [ ]:
from pathlib import Path
from getpass import getpass
from huggingface_hub import HfApi, login

adapter = Path("/content/outputs/welles/adapter")
print("files:", sorted(p.name for p in adapter.glob("*")) if adapter.exists() else "MISSING")
if not adapter.exists() or not any(adapter.iterdir()):
    raise SystemExit("Nothing to upload. Re-run cell 6 (train + upload).")

login(token=getpass("Hugging Face write token (hidden): "))
api = HfApi()
api.create_repo("n0social/welles", repo_type="model", exist_ok=True, private=False)
api.upload_folder(folder_path=str(adapter), repo_id="n0social/welles", repo_type="model")
print("https://huggingface.co/n0social/welles")